In [1]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent

sys.path.append(str(root_dir))


In [2]:
import torch
import torch.nn as nn
import gymnasium as gym
import numpy as np

from controller import Controller
from embedder import CNNVAE
from predictor import PredictorTransformer

In [3]:
from tqdm import tqdm

In [4]:
env = gym.make(
            "CarRacing-v3",
            render_mode="rgb_array",
            lap_complete_percent=0.95,
            domain_randomize=False,
            continuous=True               # <-- теперь можно одновременно делать несколько действий
        )

/Users/daniilogorodnikov/DeepZero/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [5]:
vae = CNNVAE(3, 64, 96)
vae.load_state_dict(torch.load("../embedder/vaev4.pt", map_location="cpu"))
vae.eval()

predictor = PredictorTransformer(64, 8, 128, 3, 4, 4, 128)
predictor.load_state_dict(torch.load("../predictor/predictor_ml.pt", map_location="cpu"))
predictor.eval()

controller = Controller(64, 3)

In [6]:
import cma
import copy
import torch
import numpy as np

def cma_train_controller(initial_model, env, vae, predictor,
                         sigma0=0.1, maxfevals=5000, history_len=128,
                         max_steps=900, tolfun=1e-6, verb_disp=1, seed=42):
    """
    Обучение Controller с помощью CMA-ES.
    Возвращает модель с лучшими найденными параметрами.
    """
    # 1. Получаем плоский вектор начальных параметров
    params0 = torch.nn.utils.parameters_to_vector(
        initial_model.parameters()
    ).detach().cpu().numpy()
    dim = len(params0)
    print(f"Controller parameters: {dim}")

    # 2. Функция для минимизации (чем меньше, тем лучше)
    def objective(flat_params):
        # Создаём независимую копию модели
        model = copy.deepcopy(initial_model)
        param_tensor = torch.from_numpy(flat_params).float()
        torch.nn.utils.vector_to_parameters(param_tensor, model.parameters())

        # Оценка одного эпизода
        total_reward = evaluate_controller(
            model, env, vae, predictor, history_len, max_steps
        )
        # CMA-ES минимизирует → возвращаем отрицательную награду
        return -total_reward

    # 3. Настройки CMA-ES
    opts = cma.CMAOptions()
    opts['maxfevals'] = maxfevals      # макс. число запусков симуляции
    opts['tolfun'] = tolfun            # остановка при малом изменении функции
    opts['verbose'] = verb_disp        # уровень логов: -1,0,1,2...
    opts['seed'] = seed                # для воспроизводимости
    # Можно добавить опции популяции, но CMA сама рассчитывает λ = 4 + floor(3*ln(dim))
    # Для больших размерностей это ~15-20 особей, что приемлемо.

    print(f"Starting CMA-ES | dim={dim}, sigma0={sigma0}, maxfevals={maxfevals}")
    res = cma.fmin(objective, params0, sigma0, options=opts)

    best_params = res[0]               # плоский вектор лучших параметров
    best_fitness = res[1]              # минимальное значение (отрицательная награда)

    # 4. Загружаем лучшие параметры в модель
    best_model = copy.deepcopy(initial_model)
    best_params_tensor = torch.from_numpy(best_params).float()
    torch.nn.utils.vector_to_parameters(best_params_tensor, best_model.parameters())

    print(f"CMA-ES finished. Best reward: {-best_fitness:.2f}")
    return best_model

In [7]:
def evaluate_controller(model, env, vae, predictor, history_len=128, max_steps=900):
    obs, _ = env.reset()
    observations = []
    actions = [torch.zeros(3)]

    total_reward = 0.0
    for step in range(max_steps):
        obs_tensor = torch.from_numpy(obs / 255.0).unsqueeze(0).view(1,3,96,96).float()
        with torch.no_grad():
            mu = vae.reparameterize(*vae.encode(obs_tensor))
            z_now = mu.squeeze(0)
        observations.append(z_now)

        if len(observations) > history_len:
            observations.pop(0)
            actions.pop(0)

        Z = torch.stack(observations).unsqueeze(0)
        A = torch.stack(actions).unsqueeze(0).float()
        with torch.no_grad():
            mu_next, logvar_next = predictor(Z, A)
            mu_last = mu_next[:, -1, :]
            logvar_last = logvar_next[:, -1, :]
            std = torch.exp(0.5 * logvar_last)
            eps = torch.randn_like(std)
            z_next = mu_last + eps * std

        # Действие (контроллер возвращает кортеж, берём первое)
        action = model(z_now.unsqueeze(0), z_next)[0]
        action_np = action.squeeze(0).detach().numpy()

        obs, reward, terminated, truncated, _ = env.step(action_np)
        total_reward += reward
        actions.append(action.squeeze(0))

        if terminated or truncated:
            break

    return total_reward

In [8]:
# Ваши модели мира
vae.eval()
predictor.eval()

controller = Controller(z_dim=64, action_dim=3)
best_controller = cma_train_controller(
    initial_model=controller,
    env=env,
    vae=vae,
    predictor=predictor,
    sigma0=0.1,          # начальный разброс поиска
    maxfevals=3000,      # общее число эпизодов (можно увеличить)
    history_len=128,
    max_steps=900
)

Controller parameters: 4227
Starting CMA-ES | dim=4227, sigma0=0.1, maxfevals=3000
(14_w,29)-aCMA-ES (mu_w=8.4,w_1=21%) in dimension 4227 (seed=42, Tue Jun  2 21:36:23 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     29 4.505617977528149e+01 1.0e+00 9.98e-02  1e-01  1e-01 2:46.0
    2     58 2.621359223300991e+00 1.0e+00 9.96e-02  1e-01  1e-01 5:42.8
    3     87 4.275590551181124e+01 1.0e+00 9.93e-02  1e-01  1e-01 8:34.7
    4    116 3.785276073619676e+01 1.0e+00 9.91e-02  1e-01  1e-01 11:22.2
    5    145 1.156862745098072e+01 1.0e+00 9.89e-02  1e-01  1e-01 14:13.5
    6    174 3.585987261146544e+01 1.0e+00 9.87e-02  1e-01  1e-01 16:52.4
    7    203 2.862815884476552e+01 1.0e+00 9.85e-02  1e-01  1e-01 19:33.4
    8    232 3.755244755244814e+01 1.0e+00 9.83e-02  1e-01  1e-01 22:06.8
    9    261 3.402985074626912e+01 1.0e+00 9.82e-02  1e-01  1e-01 24:40.9
   10    290 -9.333333333333421e+01 1.0e+00 9.80e-02  1e-01  1e-01 27:13.3
   11    319 -1.

KeyboardInterrupt: 

In [10]:
torch.save(best_controller.state_dict(), "controller_cmaes.pt")

In [11]:
import gymnasium as gym
import torch
import numpy as np
from torch.nn.utils import vector_to_parameters



# 2. Создаём окружение с визуализацией (человек видит окно)
env_vis = gym.make(
    "CarRacing-v3",
    render_mode="human",            # окно с графикой
    lap_complete_percent=0.95,
    domain_randomize=False,
    continuous=True
)

obs, _ = env_vis.reset()
total_reward = 0.0
done = False

# Параметры
action_dim = 3                     # steering, gas, braking
max_steps = 900                    # ограничение длины эпизода

# Инициализация истории действий и наблюдений
actions_list = [torch.zeros(action_dim, dtype=torch.float32)]
observations_list = []

step = 0
while step < max_steps and not done:
    with torch.no_grad():
        # Нормализация и подготовка наблюдения
        obs_tensor = torch.from_numpy(obs / 255.0).unsqueeze(0)
        
        # Кодирование в латентное пространство (VAE)
        z_now = vae.reparameterize(*vae.encode(obs_tensor.view(-1, 3, 96, 96).to(torch.float32)))
        observations_list.append(z_now.squeeze(0))   # убираем batch-измерение
        
        # Ограничиваем длину истории
        if len(observations_list) > 127:
            observations_list.pop(0)
            actions_list.pop(0)
        
        # Синхронизируем длины списков (действий может быть на 1 больше)
        if len(actions_list) > len(observations_list):
            actions_list = actions_list[-len(observations_list):]
        
        # Формируем тензоры для предиктора (world model)
        A = torch.stack(actions_list).unsqueeze(0) # (1, L, action_dim)
        Z = torch.stack(observations_list).unsqueeze(0) # (1, L, z_dim)
        
        # Предсказание следующего латентного состояния
        mu, logvar = predictor(Z, A)
        mu = mu[:, -1, :]            # последний предсказанный шаг
        logvar = logvar[:, -1, :]
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z_next = mu + eps * std      # сэмплирование
        
        # Контроллер выдаёт действие на основе текущего и предсказанного z
        logits = best_controller(z_now, z_now)          # (1, action_dim)
        action = logits.squeeze(0).detach().cpu()   # тензор размера (3,)
        actions_list.append(action)
    
    # Выполняем действие в среде
    obs, reward, terminated, truncated, _ = env_vis.step(action.numpy())
    total_reward += reward
    done = terminated or truncated
    step += 1

print(f"Визуализация завершена. Общая награда: {total_reward:.2f}")
env_vis.close()

KeyboardInterrupt: 

In [ ]:
env.step()


(array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        ...,
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]]], shape=(96, 96, 3), dtype=uint8),
 8.13045267489712,
 False,
 False,
 {})